In [ ]:
import pandas as pd
import numpy as np
import subprocess
subprocess.run([__import__('sys').executable, '-m', 'pip', 'install', 'lightgbm'], check=True)
import lightgbm as lgb

# ── 1. Load PFR season defense stats ────────────────────────────────────────
pfr = pd.read_csv(
    'https://github.com/nflverse/nflverse-data/releases/download/pfr_advstats/advstats_season_def.csv',
    low_memory=False
)

# ── 2. Filter to CBs 2020-2024 ───────────────────────────────────────────────
cb_pfr = pfr[
    (pfr['pos'] == 'CB') &
    (pfr['season'].between(2020, 2024)) &
    (pfr['tgt'] > 0)
].copy()

cb_pfr['incompletions']     = cb_pfr['tgt'] - cb_pfr['cmp'] - cb_pfr['int']
cb_pfr['incompletion_rate'] = cb_pfr['incompletions'] / cb_pfr['tgt']
cb_pfr['int_rate']          = cb_pfr['int']            / cb_pfr['tgt']
cb_pfr['td_rate_allowed']   = cb_pfr['td']             / cb_pfr['tgt']

# ── 3. Load snap counts ──────────────────────────────────────────────────────
snap_dfs = []
for season in [2020, 2021, 2022, 2023, 2024]:
    df = pd.read_csv(
        f'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_{season}.csv.gz',
        compression='gzip',
        low_memory=False
    )
    df['season'] = season
    snap_dfs.append(df)

snaps = pd.concat(snap_dfs, ignore_index=True)

cb_snaps = (
    snaps[snaps['position'] == 'CB']
    .groupby(['pfr_player_id', 'player'])
    .agg(career_snaps=('defense_snaps', 'sum'))
    .reset_index()
)

qualified = cb_snaps[cb_snaps['career_snaps'] >= 2500][['pfr_player_id', 'player', 'career_snaps']]

# ── 4. Aggregate PFR stats ───────────────────────────────────────────────────
cb_agg = cb_pfr.groupby(['player', 'pfr_id']).agg(
    total_targets     = ('tgt',           'sum'),
    total_incomp      = ('incompletions', 'sum'),
    total_int         = ('int',           'sum'),
    total_td          = ('td',            'sum'),
    avg_passer_rating = ('rat',           'mean'),
).reset_index()

cb_agg['incompletion_rate'] = cb_agg['total_incomp'] / cb_agg['total_targets']
cb_agg['int_rate']          = cb_agg['total_int']    / cb_agg['total_targets']
cb_agg['td_rate_allowed']   = cb_agg['total_td']     / cb_agg['total_targets']
cb_agg['passer_rating_inv'] = 158.3 - cb_agg['avg_passer_rating']

# ── 5. Filter to qualified CBs and add target rate ───────────────────────────
cb_agg = cb_agg.merge(qualified, left_on='pfr_id', right_on='pfr_player_id', how='inner')
cb_agg['target_rate']     = cb_agg['total_targets'] / cb_agg['career_snaps']
cb_agg['target_rate_inv'] = 1 - cb_agg['target_rate']

# ── 6. Assign labels ─────────────────────────────────────────────────────────
rankings = {
    'Marshon Lattimore':     1.00,
    'Jaire Alexander':       0.95,
    'Darius Slay':           0.90,
    'Patrick Surtain':       0.85,
    'Ahmad Gardner':         0.85,
    'Denzel Ward':           0.80,
    'Trent McDuffie':        0.80,
    'Stephon Gilmore':       0.75,
    'Charvarius Ward':       0.75,
    'LJarius Sneed':         0.75,
    'Jalen Ramsey':          0.70,
    'Marlon Humphrey':       0.70,
    'DJ Reed':               0.70,
    'Xavien Howard':         0.65,
    'James Bradberry':       0.65,
    'Patrick Peterson':      0.60,
    'Adoree Jackson':        0.60,
    'TreDavious White':      0.60,
    'Trevon Diggs':          0.55,
    'Tariq Woolen':          0.55,
    'Carlton Davis':         0.55,
    'Greg Newsome':          0.55,
    'AJ Terrell':            0.50,
    'Paulson Adebo':         0.50,
    'Kristian Fulton':       0.50,
    'Chidobe Awuzie':        0.45,
    'Levi Wallace':          0.45,
    'Darious Williams':      0.40,
    'Byron Murphy':          0.40,
    'Cameron Sutton':        0.40,
    'Marco Wilson':          0.40,
    'Dane Jackson':          0.40,
    'Donte Jackson':         0.40,
    'Benjamin St-Juste':     0.35,
    'Michael Davis':         0.35,
    'Taron Johnson':         0.35,
    'Alontae Taylor':        0.35,
    'Kendall Fuller':        0.35,
    'Sean Murphy-Bunting':   0.35,
    'Steven Nelson':         0.35,
    'Shaquill Griffin':      0.30,
    'Keisean Nixon':         0.30,
    'Mike Hilton':           0.30,
    'Jourdan Lewis':         0.30,
    'Fabian Moreau':         0.30,
    'Nate Hobbs':            0.25,
    'Troy Hill':             0.25,
    'Rasul Douglas':         0.25,
    'Kader Kohou':           0.20,
    'Kenny Moore':           0.20,
    'Amani Oruwariye':       0.20,
    'Asante Samuel':         0.20,
    'Ahkello Witherspoon':   0.20,
    'Tyson Campbell':        0.15,
    'Eli Apple':             0.15,
    'Michael Jackson':       0.15,
    'Jamel Dean':            0.15,
    'Ronald Darby':          0.10,
    'Desmond King':          0.10,
    'Chandon Sullivan':      0.10,
}

cb_agg['label'] = cb_agg['player_x'].map(rankings)
labeled = cb_agg[
    (cb_agg['label'].notna()) &
    (cb_agg['total_targets'] >= 50)
].copy()

# ── 7. Convert labels to integer relevance scores (required by LambdaRank) ──
# LambdaRank needs integer grades 0-4
labeled['relevance'] = (labeled['label'] * 4).round().astype(int)

# ── 8. Train LambdaRank model ────────────────────────────────────────────────
features = ['incompletion_rate', 'int_rate', 'passer_rating_inv', 'target_rate_inv']

X = labeled[features].values
y = labeled['relevance'].values

# LambdaRank needs a group array — all CBs are in one group since we rank them together
group = [len(X)]

train_data = lgb.Dataset(X, label=y, group=group)

params = {
    'objective':     'lambdarank',
    'metric':        'ndcg',
    'ndcg_eval_at':  [10],
    'learning_rate': 0.05,
    'num_leaves':    8,
    'min_data_in_leaf': 1,
    'verbose':       -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=200,
)

# ── 9. Print feature importances as weightages ───────────────────────────────
importance = model.feature_importance(importance_type='gain')
total = importance.sum()
weightages = importance / total * 100

print("\nFeature importances (weightages):")
for feat, w in zip(features, weightages):
    print(f"  {feat}: {w:.1f}%")

# ── 10. Score and rank all qualified CBs ─────────────────────────────────────
cb_agg_filtered = cb_agg[cb_agg['total_targets'] >= 50].copy()

X_all = cb_agg_filtered[features].values
cb_agg_filtered['wcs'] = model.predict(X_all)

cb_ranked = cb_agg_filtered[['player_x', 'total_targets', 'incompletion_rate', 'int_rate', 'target_rate', 'avg_passer_rating', 'wcs', 'label']]\
    .sort_values('wcs', ascending=False)\
    .reset_index(drop=True)

cb_ranked.index += 1
print(f"\nFull CB rankings:")
print(cb_ranked.to_string())

cb_ranked.to_csv('cb_rankings.csv')